# Tiamat — Relatório científico e preparação da entrega

Reunião de interpretações em um relatório científico navegável. As células abaixo exibem resultados.


In [1]:
from pathlib import Path
import hashlib
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import optuna
from IPython.display import display, Markdown
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)

SEED = 210906
LANCE = 40
MIN_CONFIGURACOES = 100
MODELOS = ['Regressao_Logistica', 'Random_Forest', 'Gradient_Boosting', 'XGBoost', 'SVM']
ATRIBUTOS = [
    'rank_b', 'rank_w', 'komi', 'handicap',
    'maior_grupo_b', 'maior_grupo_w',
    'liberdades_maior_grupo_b', 'liberdades_maior_grupo_w',
    'pedras_em_jogo_b', 'pedras_em_jogo_w',
    'pedras_capturadas_por_b', 'pedras_capturadas_por_w',
    'distancia_media_tengen_b', 'distancia_media_tengen_w',
    'quantidade_grupos_b', 'quantidade_grupos_w',
    'tamanho_medio_grupos_b', 'tamanho_medio_grupos_w',
    'liberdades_medias_grupos_b', 'liberdades_medias_grupos_w',
    'grupos_em_atari_b', 'grupos_em_atari_w',
    'pedras_em_atari_b', 'pedras_em_atari_w'
]
CANDIDATOS = [Path('data/piloto_final/lance_040.csv'),
    Path('data/piloto_corrigido/lance_040.csv'), Path('data/piloto/lance_040.csv'),
    Path('data/processed/lance_040.csv'), Path('lance_040.csv')]
ARQUIVO_CSV = next((p for p in CANDIDATOS if p.exists()), None)
if ARQUIVO_CSV is None:
    raise FileNotFoundError('Ajuste CANDIDATOS para localizar o mesmo CSV usado no notebook 03.')
PASTA = Path('results/lance_040')
PASTA.mkdir(parents=True, exist_ok=True)
FIGURAS = PASTA / 'figuras'
FIGURAS.mkdir(exist_ok=True)

def sha256_arquivo(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def ler_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def salvar_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2,
        allow_nan=False), encoding='utf-8')

def carregar_dados(split):
    dados = pd.read_csv(ARQUIVO_CSV)
    dados = dados.loc[dados['split'].eq(split)].copy()
    if not len(dados):
        raise ValueError(f'Conjunto {split} vazio.')
    assert dados.game_id.is_unique
    assert dados.lance.eq(LANCE).all()
    assert dados.vencedor.isin(['B', 'W']).all()
    return dados

def separar_xy(dados):
    X = dados[ATRIBUTOS].apply(pd.to_numeric, errors='raise')
    y = dados.vencedor.map({'B': 0, 'W': 1}).astype(int)
    assert not np.isinf(X.to_numpy()).any()
    return X, y

def pontuacoes(modelo, X):
    if hasattr(modelo, 'predict_proba'):
        classe_w = list(modelo.classes_).index(1)
        return modelo.predict_proba(X)[:, classe_w], 'probabilidade_W'
    return np.asarray(modelo.decision_function(X)).ravel(), 'margem_W'

def metricas_predicoes(y, pred, score):
    y = np.asarray(y)
    return {'n': int(len(y)), 'accuracy': float(accuracy_score(y, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y, pred)),
        'f1_macro': float(f1_score(y, pred, average='macro', zero_division=0)),
        'roc_auc': float(roc_auc_score(y, score)) if len(np.unique(y)) == 2 else np.nan}

def avaliar_modelo(modelo, dados):
    X, y = separar_xy(dados)
    pred = modelo.predict(X)
    score, tipo = pontuacoes(modelo, X)
    tabela = pd.DataFrame({'game_id': dados.game_id.to_numpy(),
        'y_real': y.to_numpy(), 'y_pred': pred, 'score_W': score,
        'tipo_score': tipo, 'acertou': pred == y.to_numpy()})
    return metricas_predicoes(y, pred, score), tabela

def carregar_estudos():
    estudos, linhas = {}, []
    for nome in MODELOS:
        db = (PASTA / f'optuna_{nome}.db').resolve()
        distintos = completos = 0
        melhor = np.nan
        if db.exists():
            # Mesmo nome de estudo e formato SQLite do notebook 03 antigo.
            estudo = optuna.load_study(study_name=f'tiamat_lance40_{nome}',
                storage=f'sqlite:///{db.as_posix()}')
            estudos[nome] = estudo
            trials = [t for t in estudo.trials
                if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None]
            completos = len(trials)
            distintos = len({json.dumps(t.params, sort_keys=True) for t in trials})
            if completos:
                melhor = float(estudo.best_value)
        resumo_path = PASTA / f'resumo_{nome}.json'
        if resumo_path.exists():
            resumo = ler_json(resumo_path)
            if (resumo.get('seed') != SEED or resumo.get('lance') != LANCE
                    or resumo.get('folds') != 5):
                raise ValueError(f'Resumo de {nome} incompatível com o protocolo do 03 antigo.')
        linhas.append({'modelo': nome, 'configuracoes_distintas': distintos,
            'trials_completos': completos, 'roc_auc_cv': melhor,
            'pronto': distintos >= MIN_CONFIGURACOES})
    return estudos, pd.DataFrame(linhas)

def carregar_selecionado():
    selecao = ler_json(PASTA / 'selecao_congelada.json')
    if selecao.get('origem') != '03_antigo':
        raise ValueError('A seleção existente não foi produzida por esta sequência compatível.')
    if selecao['sha256_csv'] != sha256_arquivo(ARQUIVO_CSV):
        raise ValueError('O CSV foi alterado após a seleção. Preserve os dados desta execução.')
    path = PASTA / selecao['arquivo_modelo']
    if selecao['sha256_modelo'] != sha256_arquivo(path):
        raise ValueError('O modelo selecionado foi alterado após o congelamento.')
    return joblib.load(path), selecao

print('CSV:', ARQUIVO_CSV, '| resultados:', PASTA)
print('SEED:', SEED, '| 24 atributos | compatível com o notebook 03 original')


CSV: data\piloto_final\lance_040.csv | resultados: results\lance_040
SEED: 210906 | 24 atributos | compatível com o notebook 03 original


## 1. Identificação e pergunta de pesquisa

Autor: Pedro Lucas M. S. Sá — Ilum Escola de Ciência, CNPEM.

Disciplina: Aprendizado de Máquina — Docente: Prof. Dr. Daniel R. Cassar.

O trabalho investiga a previsão da cor vencedora utilizando informações disponíveis após as primeiras 40 jogadas.

A pergunta de pesquisa é: até que ponto as condições iniciais e as características dessa posição permitem discriminar vitórias das pretas e das brancas?

## 2. Dados e engenharia de atributos

Os registros SGF são provenientes do Joe’s Go Database, que disponibiliza partidas separadas em treino, validação e teste.
A amostra filtrada contém 5.156 partidas, sendo 4.612 de treino, 270 de validação e 274 de teste; foram excluídas duplicatas, partidas curtas, resultados sem vencedor e ranks ausentes ou incomuns.
Os 24 atributos incluem ranks, komi, handicap, pedras, capturas, grupos, liberdades, distância ao tengen e situações de atari; a escala de ranks segue zk → −z, xd → x e yp → y+6.
O komi representa compensação em pontos e o handicap, vantagem inicial; passes contam como jogadas, pedras iniciais não entram nessa contagem e metadados de encerramento ficam fora dos preditores.


In [2]:
train = carregar_dados('train')
print('Partidas do treino:', len(train), '| atributos:', len(ATRIBUTOS))
display(train[ATRIBUTOS].isna().sum().rename('ausentes_treino').to_frame())
display(pd.DataFrame({'atributo': ATRIBUTOS}))


Partidas do treino: 4612 | atributos: 24


,ausentes_treino
rank_b,0
rank_w,0
komi,65
handicap,2
maior_grupo_b,0
maior_grupo_w,0
liberdades_maior_grupo_b,0
liberdades_maior_grupo_w,0
pedras_em_jogo_b,0
pedras_em_jogo_w,0


,atributo
0,rank_b
1,rank_w
2,komi
3,handicap
4,maior_grupo_b
5,maior_grupo_w
6,liberdades_maior_grupo_b
7,liberdades_maior_grupo_w
8,pedras_em_jogo_b
9,pedras_em_jogo_w


## 3. Métodos

Foram comparados Regressão Logística, Random Forest, Gradient Boosting, XGBoost e SVM, com 100 configurações distintas completas por algoritmo, além do DummyClassifier como referência.
A otimização utilizou Optuna e maximizou a ROC-AUC média de cinco folds estratificados, com semente 210906; accuracy, balanced accuracy e F1-macro complementaram a avaliação.
Imputação por mediana, padronização para Regressão Logística e SVM e seleção de 10, 15, 20 ou 24 atributos foram ajustadas dentro dos pipelines e dos folds, seguindo as boas práticas de prevenção de vazamento.
Os melhores pipelines foram ajustados no treino e comparados em val; o escolhido permaneceu treinado somente em train e foi congelado antes da avaliação no teste.


In [5]:
estudos, status = carregar_estudos()
display(Markdown('### Estudos do notebook 03'))
display(status)
for titulo, nome in {
    'Resultados CV registrados no 03': 'comparacao_cv.csv',
    'Comparação na validação': 'comparacao_validacao.csv',
    'Importância por permutação': 'importancia_permutacao_val.csv',
    'Diagnóstico por subgrupo': 'diagnostico_subgrupos_val.csv',
}.items():
    display(Markdown(f'### {titulo}'))
    path = PASTA / nome
    if path.exists():
        display(pd.read_csv(path).head(30))
    else:
        print('Ainda não produzido:', nome)
for nome, estudo in estudos.items():
    if any(t.state == optuna.trial.TrialState.COMPLETE for t in estudo.trials):
        display(Markdown(f'### Melhores parâmetros — {nome}'))
        display(pd.Series(estudo.best_params, name='valor').to_frame())
if (PASTA / 'avaliacao_final_teste.json').exists():
    final = ler_json(PASTA / 'avaliacao_final_teste.json')
    display(Markdown('### Avaliação final registrada'))
    display(pd.DataFrame({'selecionado': final['metricas'], 'Dummy': final['dummy']}))
    display(pd.DataFrame(final['intervalos_bootstrap']).T)
else:
    print('Avaliação final ainda não realizada.')
print('Consulte as saídas do notebook 02 para discutir os baselines: a versão antiga não exporta baselines_folds.csv.')


### Estudos do notebook 03

,modelo,configuracoes_distintas,trials_completos,roc_auc_cv,pronto
0,Regressao_Logistica,100,100,0.562428,True
1,Random_Forest,100,100,0.555193,True
2,Gradient_Boosting,100,100,0.563409,True
3,XGBoost,100,100,0.562010,True
4,SVM,100,100,0.561449,True


### Resultados CV registrados no 03

,modelo,roc_auc_cv,dp_roc_auc_cv,balanced_accuracy_cv
0,Regressao_Logistica,0.562428,0.021505,0.546533
1,Random_Forest,0.555193,0.017908,0.549283
2,Gradient_Boosting,0.563409,0.012874,0.544721
3,XGBoost,0.562010,0.014717,0.549721
4,SVM,0.561449,0.021433,0.544680


### Comparação na validação

,modelo,n,accuracy,balanced_accuracy,f1_macro,roc_auc,roc_auc_cv,fit_treino_s
0,Regressao_Logistica,270,0.555556,0.555678,0.555556,0.575654,0.562428,0.237866
1,Random_Forest,270,0.577778,0.577795,0.577755,0.608254,0.555193,10.772033
2,Gradient_Boosting,270,0.559259,0.558778,0.558526,0.562346,0.563409,2.764117
3,XGBoost,270,0.518519,0.518413,0.518413,0.568684,0.562010,0.495264
4,SVM,270,0.540741,0.540640,0.540640,0.565940,0.561449,12.793491
5,Dummy_prior,270,0.492593,0.500000,0.330025,0.500000,0.500000,NaN


... (14 outputs left out)

### Avaliação final registrada

,selecionado,Dummy
n,274.000000,274.000000
accuracy,0.558394,0.525547
balanced_accuracy,0.557425,0.500000
f1_macro,0.557398,0.344498
roc_auc,0.579167,0.500000


,estimativa,inferior_95,superior_95
accuracy,0.558394,0.500000,0.620438
balanced_accuracy,0.557425,0.498495,0.618323
f1_macro,0.557398,0.498486,0.618405
roc_auc,0.579167,0.513882,0.645943


Consulte as saídas do notebook 02 para discutir os baselines: a versão antiga não exporta baselines_folds.csv.


## 4. Resultados e discussão

### 4.1. Análise exploratória

O treino apresentou 2.271 vitórias das pretas e 2.341 das brancas, correspondendo a classes aproximadamente equilibradas.
Foram identificados 65 valores ausentes de komi e dois de handicap, tratados por imputação nos pipelines.
A correlação entre pedras brancas presentes e capturas pelas pretas foi aproximadamente −0,9987, compatível com a retirada de pedras pelas capturas.
Também foram observadas correlações elevadas entre atributos de grupos e atari; os 24 atributos foram mantidos para a seleção controlada durante a otimização.

### 4.2. Baselines e otimização

O Dummy apresentou ROC-AUC de 0,5000, enquanto a regressão logística inicial alcançou 0,5618, indicando capacidade discriminativa ainda modesta.
O Gradient Boosting obteve a maior média de CV, 0,5634, apenas cerca de 0,0016 acima da regressão logística inicial.
Na validação, o Random Forest liderou com 0,6083, superando o Dummy em 0,1083 e a regressão logística otimizada em 0,0326.
Todos os melhores pipelines preservaram 24 atributos; a mudança do ranking reforça que o melhor resultado da busca não constitui uma avaliação independente.

### 4.3. Explicabilidade e erros

A importância por permutação avaliou a queda da ROC-AUC ao embaralhar cada atributo, mantendo o modelo fixo, conforme o procedimento descrito pelo scikit-learn.
Komi e rank das brancas apresentaram as maiores quedas médias: 0,0706 e 0,0293, respectivamente, indicando dependência preditiva dessas informações.
Na validação ocorreram 156 acertos e 114 erros; o subgrupo com handicap apresentou AUC de 0,4949, com apenas 29 partidas.
Atributos correlacionados e grupos pequenos limitam a interpretação; as importâncias descrevem o comportamento do pipeline e não demonstram causalidade.

### 4.4. Avaliação final no teste

Nas 274 partidas de teste, o Random Forest apresentou ROC-AUC de 0,5792, com intervalo bootstrap de 95% entre 0,5139 e 0,6459.
A accuracy foi 0,5584, a balanced accuracy 0,5574 e o F1-macro 0,5574; o Dummy apresentou accuracy de 0,5255 e ROC-AUC de 0,5000.
A ROC-AUC ficou abaixo dos 0,6083 da validação, mantendo um resultado de discriminação modesta na amostra final.
Embora o intervalo da AUC tenha excluído 0,5, o da balanced accuracy incluiu esse referencial, mostrando incerteza na classificação com o limiar adotado.


## 5. Limitações e trabalhos futuros

Os resultados representam uma amostra do JGDB e informações disponíveis no lance 40, restringindo sua generalização para outros contextos.
A codificação dos ranks, as simplificações da reconstrução, os atributos correlacionados e a possível dependência entre partidas dos mesmos jogadores permanecem como limitações.
O bootstrap utilizou 2.000 reamostragens estratificadas de predições fixas, sem incorporar a busca de hiperparâmetros, o retreinamento ou a dependência entre partidas.
Trabalhos futuros podem ampliar a amostra e comparar os lances 0 e 120, utilizando uma avaliação independente para as novas decisões.

## Conclusões e principais aprendizados

Os atributos do lance 40 forneceram informação preditiva modesta sobre o vencedor das partidas avaliadas.
O Random Forest, escolhido na validação, alcançou ROC-AUC de 0,5792 e accuracy de 55,84% no teste, superando pontualmente o Dummy.
A mudança do ranking entre CV e validação destacou a importância de separar otimização, seleção e avaliação final.
O trabalho também evidenciou a necessidade de verificar os dados, interpretar a seleção de atributos com cautela e distinguir importância preditiva de causalidade.


## Descrição do uso de IA neste trabalho

O ChatGPT auxiliou no planejamento, na organização dos notebooks e na elaboração e adaptação dos códigos. link: https://chatgpt.com/share/6af5a137-1360-83e9-a134-743aff35f281


## Referências consultadas

- Cassar, D. R. *Criaturas Lendárias do Reino de Lumi e onde encontrá-las*, seção 4, Tiamat, pp. 7–9. Material da ATP-203 fornecido no projeto.
- [scikit-learn: prevenção de vazamento e pipelines](https://scikit-learn.org/stable/common_pitfalls.html).
- [scikit-learn: validação cruzada](https://scikit-learn.org/stable/modules/cross_validation.html).
- [scikit-learn: métricas de classificação](https://scikit-learn.org/stable/modules/model_evaluation.html).
- [Optuna: estudos persistentes](https://optuna.readthedocs.io/en/stable/tutorial/20_recipes/001_rdb.html).
- [Optuna: Study e estados dos trials](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.Study.html).
- [scikit-learn: permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html).
- [scikit-learn: dependência parcial](https://scikit-learn.org/stable/modules/partial_dependence.html).
- [scikit-learn: SelectKBest](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html).
- [XGBoost: hiperparâmetros](https://xgboost.readthedocs.io/en/stable/parameter.html).

Consulta: 11/11/2026. Complete a referência da fonte original dos SGFs com o endereço e a data de obtenção realmente utilizados.


## Verificação da entrega

A verificação automática abaixo identifica arquivos produzidos. Ela não atribui nota e não verifica qualidade da discussão, compreensão dos conceitos ou autoria. Leia seu texto e execute os notebooks antes da apresentação.


In [4]:
esperados = {
    'Notebooks originais de EDA e baselines': [Path('01_analise_exploratoria_tiamat.ipynb'),
        Path('02_baselines_preprocessamento_tiamat.ipynb')],
    'Cinco estudos do 03': [PASTA / f'optuna_{n}.db' for n in MODELOS],
    'Seleção em val': [PASTA / 'comparacao_validacao.csv', PASTA / 'selecao_congelada.json'],
    'Explicabilidade': [PASTA / 'importancia_permutacao_val.csv'],
    'Teste final': [PASTA / 'avaliacao_final_teste.json', PASTA / 'predicoes_teste_final.csv'],
}
linhas = []
for etapa, caminhos in esperados.items():
    faltantes = [str(p) for p in caminhos if not p.exists()]
    linhas.append({'etapa': etapa, 'arquivos_presentes': not faltantes,
        'faltantes': ', '.join(faltantes)})
display(pd.DataFrame(linhas))
_, status = carregar_estudos()
display(status)
print('100 configurações distintas completas nos cinco algoritmos:', bool(status.pronto.all()))
for nome in ['01_analise_exploratoria_tiamat.ipynb', '02_baselines_preprocessamento_tiamat.ipynb']:
    if Path(nome).exists():
        n = ler_json(nome)
        c = [c for c in n['cells'] if c['cell_type'] == 'code']
        print(nome, '| células com execução registrada:',
            sum(c.get('execution_count') is not None for c in c), '/', len(c))
print('A presença de arquivos ou saídas não verifica a qualidade da discussão nem a conclusão dos experimentos.')
print('Figuras produzidas por 04–06:', len(list(FIGURAS.glob('*.png'))))


,etapa,arquivos_presentes,faltantes
0,Notebooks originais de EDA e baselines,True,
1,Cinco estudos do 03,True,
2,Seleção em val,True,
3,Explicabilidade,True,
4,Teste final,True,


,modelo,configuracoes_distintas,trials_completos,roc_auc_cv,pronto
0,Regressao_Logistica,100,100,0.562428,True
1,Random_Forest,100,100,0.555193,True
2,Gradient_Boosting,100,100,0.563409,True
3,XGBoost,100,100,0.562010,True
4,SVM,100,100,0.561449,True


100 configurações distintas completas nos cinco algoritmos: True
01_analise_exploratoria_tiamat.ipynb | células com execução registrada: 7 / 7
02_baselines_preprocessamento_tiamat.ipynb | células com execução registrada: 6 / 6
A presença de arquivos ou saídas não verifica a qualidade da discussão nem a conclusão dos experimentos.
Figuras produzidas por 04–06: 6


**Antes de entregar:** complete identificação e referência dos dados; revise os campos “Sua escrita”; confira títulos e eixos das figuras; inclua README, dependências, os notebooks originais e os atualizados, resultados e discussões no repositório; remova caches. Se usou HPC, inclua a saída real do SLURM. Prepare-se para explicar as escolhas e os códigos.

**Para retomarmos:** envie os notebooks com suas interpretações. A revisão poderá verificar coerência entre texto e resultados, corrigir conceitos e sugerir ajustes sem substituir sua escrita.
